# Phase 5 -- Layer-Specific SMI Effects, Trial-Count-Matched (DREADD saline/DCZ cohort)

Mirrors `5.LayerSpecific.py`'s core layer/depth-group analysis, but consumes `4.SessionComparison_TrialMatched.ipynb`'s per-cell tables (`{group}_trial_matched_comparison_table.csv`) instead of Phase 4's regular ones -- so `condition` here is `'saline'`, `'dcz_block_1'`, ..., `'dcz_block_n'` (per group, however many blocks that group's dcz session split into) rather than `'baseline'`/`'saline'`/`'dcz'`.

**Every function below that operates on pooled cells (the `compare_smi_by_layer`/`compare_smi_by_depth_group`/`test_layer_depth_interaction` family) is reused essentially unchanged from `5.LayerSpecific.py`** -- they're already generic over whatever categories are in `condition`, so no logic changes were needed, only the condition ordering/coloring used for plots (`_condition_order_and_colors`, saline = orange, `dcz_block_N` = a Purples gradient by block number, same convention `4.SessionComparison_TrialMatched.ipynb`'s own `plot_smi_across_blocks` already uses).

**Pseudo-replication caveat -- read before trusting any p-value here.** `5.LayerSpecific.py`'s own docstring already flags that its pooled-cell tests (5.1/5.2/5.4-equivalents) treat cells within one group as independent, when really there's only one saline session and one paired dcz session per group -- a known pseudo-replication problem, later corrected there by Functions 5.11-5.14 (paired test across the independent comparison GROUPS instead of pooled cells). **That problem is worse here**: the dcz blocks within one group aren't even separate sessions -- they're overlapping/adjacent trial-slices of the exact same recording (see `4.SessionComparison_TrialMatched.ipynb`'s block-construction note: the last two blocks can share most of their trials). So treat every per-group, per-layer result below as descriptive/exploratory, not a standalone significance claim -- useful for looking at trends (does a layer's SMI drift block-to-block within a session?), not for a confirmatory p-value. A proper paired-across-groups version (mirroring 5.11-5.14, with "block position" as the within-subject factor and comparison group as the independent unit) would be the right next step before treating anything here as confirmatory -- not built yet, scoped out of this first pass the same way 5.11-5.14 was itself added to `5.LayerSpecific.py` only after 5.1-5.10 was already working.

In [54]:
import sys
sys.path.insert(0, r"C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation")

import os
import glob
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Qt5Agg')  # for plt.show() popups
import matplotlib.pyplot as plt
from matplotlib import rcParams
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy.stats import kruskal, mannwhitneyu, rankdata

rcParams['legend.fontsize'] = 20
rcParams['axes.labelsize'] = 20
rcParams['axes.titlesize'] = 25
rcParams['xtick.labelsize'] = 20
rcParams['ytick.labelsize'] = 20

# Point this at whichever animal you're processing -- kept for both so
# switching doesn't leave other TEST_* constants pointing at the wrong
# animal (same fix 4.SessionComparison_TrialMatched.ipynb needed).
TEST_ANIMAL_DIR_JSY090 = r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD"
TEST_ANIMAL_DIR_JSY093 = r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD"
TEST_ANIMAL_DIR = TEST_ANIMAL_DIR_JSY093

## Setup -- `load_all_group_dfs_from_trial_matched`

Loads every `{group}_trial_matched_comparison_table.csv` already saved by `4.SessionComparison_TrialMatched.ipynb`'s `save_trial_matched_outputs` -- same idea as `5.LayerSpecific.py`'s `load_all_group_dfs_from_phase4`, just pointed at the trial-matched output directory/filename pattern instead.

In [55]:
def load_all_group_dfs_from_trial_matched(output_dir):
    """
    Load every group's per-cell table already saved by
    4.SessionComparison_TrialMatched.ipynb's save_trial_matched_outputs
    ('{group_name}_trial_matched_comparison_table.csv').

    Parameters
    ----------
    output_dir : str
        e.g. os.path.join(ANIMAL_DIR, 'Phase4_TrialMatched_Results').

    Returns
    -------
    all_group_dfs : dict
        {group_name: df}.
    """
    csv_paths = sorted(glob.glob(os.path.join(output_dir, '*_trial_matched_comparison_table.csv')))

    if not csv_paths:
        raise FileNotFoundError(f"No *_trial_matched_comparison_table.csv found in {output_dir} -- "
                                 "has 4.SessionComparison_TrialMatched.ipynb's save step been run for this animal?")

    all_group_dfs = {}
    for csv_path in csv_paths:
        group_name = os.path.basename(csv_path)[:-len('_trial_matched_comparison_table.csv')]
        df = pd.read_csv(csv_path)
        all_group_dfs[group_name] = df
        print(f"Loaded '{group_name}': {len(df)} cell-rows <- {csv_path}")

    print(f"\nLoaded {len(all_group_dfs)} group(s) from {output_dir}: {list(all_group_dfs.keys())}")
    return all_group_dfs

## Function -- `compare_smi_across_conditions`

Reimplemented unchanged from `5.LayerSpecific.py`'s Function 5.1 -- Kruskal-Wallis omnibus + pairwise Mann-Whitney U (Holm-corrected), across whatever categories are present. Fully generic already, no adaptation needed for `dcz_block_N`-style categories.

In [56]:
def compare_smi_across_conditions(df, group_col='condition', value_col='SMI', filter_col='valid'):
    """
    Kruskal-Wallis omnibus + pairwise Mann-Whitney U (Holm-corrected)
    across whatever categories are present.

    Parameters
    ----------
    df : pandas.DataFrame
        One group's or one layer's/depth-group's table.
    group_col, value_col, filter_col : str

    Returns
    -------
    result : dict or None
        None (with a printed message) if fewer than 2 categories remain
        after filtering. Otherwise:
        {'group_medians': {cat: median}, 'group_n': {cat: n},
         'omnibus_stat', 'omnibus_p',
         'pairwise': DataFrame(cond_a, cond_b, median_diff, U_stat, p_raw, p_holm)}.
    """
    filtered = df[df[filter_col]]
    categories = [c for c in filtered[group_col].unique() if pd.notna(c)]

    if len(categories) < 2:
        print(f"Only {len(categories)} category(ies) present after filtering on '{filter_col}' "
              f"-- nothing to compare ({categories}).")
        return None

    samples = {cat: filtered.loc[filtered[group_col] == cat, value_col].to_numpy()
               for cat in categories}

    group_medians = {cat: float(np.median(vals)) for cat, vals in samples.items()}
    group_n = {cat: len(vals) for cat, vals in samples.items()}

    omnibus_stat, omnibus_p = kruskal(*samples.values())

    pairwise_rows = []
    for cat_a, cat_b in combinations(categories, 2):
        u_stat, p_raw = mannwhitneyu(samples[cat_a], samples[cat_b], alternative='two-sided')
        pairwise_rows.append({
            'cond_a': cat_a, 'cond_b': cat_b,
            'median_diff': group_medians[cat_a] - group_medians[cat_b],
            'U_stat': u_stat, 'p_raw': p_raw,
        })

    pairwise_df = pd.DataFrame(pairwise_rows)
    if len(pairwise_df) > 0:
        _, p_holm, _, _ = multipletests(pairwise_df['p_raw'], method='holm')
        pairwise_df['p_holm'] = p_holm

    print(f"Categories ({group_col}): {categories}")
    print(f"  n per category: {group_n}")
    print(f"  median {value_col} per category: {group_medians}")
    print(f"  Kruskal-Wallis: H={omnibus_stat:.3f}, p={omnibus_p:.4f}")
    print(f"\n  Pairwise (Holm-corrected):")
    print(pairwise_df.to_string(index=False))

    return {
        'group_medians': group_medians,
        'group_n': group_n,
        'omnibus_stat': omnibus_stat,
        'omnibus_p': omnibus_p,
        'pairwise': pairwise_df,
    }

## Function -- `compare_smi_by_layer`

Reimplemented unchanged from `5.LayerSpecific.py`'s Function 5.2 -- loops `compare_smi_across_conditions` over each layer present (canonical L2/3 -> L4 -> L5 -> L6 order). Caveat carried over unchanged: each layer's Holm correction applies only within that layer's own pairwise tests, not across layers.

In [57]:
CANONICAL_LAYER_ORDER = ['L2/3', 'L4', 'L5', 'L6']


def _layer_order(layers_present):
    return ([l for l in CANONICAL_LAYER_ORDER if l in layers_present]
            + [l for l in layers_present if l not in CANONICAL_LAYER_ORDER])


def compare_smi_by_layer(df, layer_col='layer', group_col='condition', value_col='SMI', filter_col='valid'):
    """
    Loop compare_smi_across_conditions over each layer present.

    Parameters
    ----------
    df : pandas.DataFrame
        One group's table.
    layer_col, group_col, value_col, filter_col : str

    Returns
    -------
    results : dict
        {layer: compare_smi_across_conditions(...) result or None}.
    summary_df : pandas.DataFrame
        One row per (layer, pairwise comparison) that had 2+ categories.
    """
    layers_present = [l for l in df[layer_col].dropna().unique()]
    layer_order = _layer_order(layers_present)

    results = {}
    summary_rows = []
    for layer in layer_order:
        print(f"\n--- Layer {layer} ---")
        layer_df = df[df[layer_col] == layer]
        result = compare_smi_across_conditions(layer_df, group_col=group_col, value_col=value_col,
                                                filter_col=filter_col)
        results[layer] = result
        if result is not None:
            for _, row in result['pairwise'].iterrows():
                summary_rows.append({
                    'layer': layer, 'cond_a': row['cond_a'], 'cond_b': row['cond_b'],
                    'median_diff': row['median_diff'], 'p_holm': row['p_holm'],
                })

    summary_df = pd.DataFrame(summary_rows)
    if len(summary_df) > 0:
        print("\n=== Summary across layers (each layer's own Holm correction -- not corrected across layers) ===")
        print(summary_df.to_string(index=False))

    return results, summary_df

## Function -- `_condition_order_and_colors`

The one piece that actually needed adapting from `5.LayerSpecific.py`: its plotting functions hardcode `['baseline', 'saline', 'dcz']` ordering/colors, which doesn't fit `dcz_block_1..n`. This orders saline first, then `dcz_block_N` by block number, with the same orange/Purples-gradient scheme `4.SessionComparison_TrialMatched.ipynb`'s `plot_smi_across_blocks` already uses -- so figures from both notebooks read consistently.

In [58]:
def _condition_order_and_colors(conditions_present):
    """
    Order + color scheme for whatever's in the condition column: saline
    first (if present), then every other condition -- sorted numerically
    if its name ends in '_<number>' (e.g. dcz_block_1, dcz_block_2, ...),
    alphabetically otherwise (e.g. dcz_random_subsampled) -- each given
    its own shade from a Purples gradient. Generic over which trial-
    matching method produced the data, not hardcoded to 'dcz_block_N'
    naming specifically.

    Parameters
    ----------
    conditions_present : iterable of str

    Returns
    -------
    order : list of str
    color_by_category : dict
    """
    conditions_present = list(conditions_present)

    def _sort_key(cond):
        cond_str = str(cond)
        if '_' in cond_str:
            suffix = cond_str.rsplit('_', 1)[1]
            if suffix.isdigit():
                return (0, int(suffix), cond_str)
        return (1, 0, cond_str)

    order = ['saline'] if 'saline' in conditions_present else []
    others = sorted([c for c in conditions_present if c != 'saline'], key=_sort_key)
    order += others

    color_by_category = {'saline': 'tab:orange'}
    n_others = max(len(others), 1)
    other_colors = plt.cm.Purples(np.linspace(0.4, 0.9, n_others))
    for i, cond in enumerate(others):
        color_by_category[cond] = other_colors[i]

    return order, color_by_category

## Function -- `plot_smi_by_layer`

Adapted from `5.LayerSpecific.py`'s Function 5.3 -- same grid-of-violins-per-layer design, using `_condition_order_and_colors` instead of the hardcoded baseline/saline/dcz scheme.

In [59]:
def _p_to_stars(p):
    """Convert a p-value to a significance-star string ('ns' if >= .05)."""
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    return 'ns'


def _annotate_saline_vs_block_brackets(ax, category_order, data_by_category, stats_rows):
    """
    Draw saline-vs-each-dcz_block_N significance brackets above one violin
    panel. Only saline-vs-block comparisons are ever drawn here (never
    block-vs-block) -- matching compare_saline_vs_each_dcz_block's scoped
    Holm correction, not compare_smi_across_conditions's full pairwise
    table.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
    category_order : list of str
        x-axis category order (1-indexed positions in the violinplot).
    data_by_category : list of np.ndarray
        Same order as category_order, used to find headroom above the data.
    stats_rows : pandas.DataFrame or None
        Rows with 'dcz_block' and 'p_holm' columns for this panel only
        (already filtered to whatever layer/depth_group this axis shows).

    Returns
    -------
    y_top : float
        Highest y-coordinate used by a bracket (or the data max if none
        were drawn), so the caller can extend ylim to fit.
    """
    data_max = max((v.max() for v in data_by_category if len(v)), default=1.0)

    if 'saline' not in category_order or stats_rows is None or len(stats_rows) == 0:
        return data_max

    x_saline = category_order.index('saline') + 1
    y0 = max(data_max, 1.0) + 0.15
    step = 0.13
    tick = 0.03

    y_top = data_max
    i = 0
    for _, row in stats_rows.iterrows():
        block = row['dcz_block']
        if block not in category_order:
            continue
        x_block = category_order.index(block) + 1
        y = y0 + step * i
        i += 1
        ax.plot([x_saline, x_saline, x_block, x_block],
                [y - tick, y, y, y - tick], color='black', lw=1)
        ax.text((x_saline + x_block) / 2, y + 0.01, _p_to_stars(row['p_holm']),
                ha='center', va='bottom', fontsize=11)
        y_top = max(y_top, y)

    return y_top


def plot_smi_by_layer(df, layer_col='layer', group_col='condition', value_col='SMI', filter_col='valid', title='',
                      stats_df=None):
    """
    Grid of violin+strip plots, one panel per layer present.

    Parameters
    ----------
    df : pandas.DataFrame
    layer_col, group_col, value_col, filter_col : str
    title : str
    stats_df : pandas.DataFrame, optional
        compare_saline_vs_each_dcz_block_by_layer's output -- if given,
        draws a saline-vs-each-dcz_block_N significance bracket (stars
        from Holm-corrected p, 'ns' if not significant) on each panel
        that has both 'saline' and that block.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    filtered = df[df[filter_col]]
    layers_present = [l for l in filtered[layer_col].dropna().unique()]
    layer_order = _layer_order(layers_present)

    n_layers = len(layer_order)
    n_cols = 2
    n_rows = int(np.ceil(n_layers / n_cols)) if n_layers > 0 else 1
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(8 * n_cols, 7 * n_rows))
    axes = np.atleast_1d(axes).flatten()

    for ax, layer in zip(axes, layer_order):
        layer_df = filtered[filtered[layer_col] == layer]
        category_order, color_by_category = _condition_order_and_colors(layer_df[group_col].unique())

        data_by_category = [layer_df.loc[layer_df[group_col] == cat, value_col].to_numpy()
                            for cat in category_order]

        if len(data_by_category) == 0 or all(len(d) == 0 for d in data_by_category):
            ax.set_title(f"{layer} (no data)")
            ax.axis('off')
            continue

        parts = ax.violinplot(data_by_category, showmedians=True)
        for i, body in enumerate(parts['bodies']):
            body.set_facecolor(color_by_category.get(category_order[i], 'gray'))
            body.set_alpha(0.4)

        rng = np.random.default_rng(0)
        for i, vals in enumerate(data_by_category):
            jitter = rng.uniform(-0.08, 0.08, size=len(vals))
            ax.scatter(np.full(len(vals), i + 1) + jitter, vals,
                       color=color_by_category.get(category_order[i], 'gray'), s=12, alpha=0.5)

        if stats_df is not None and len(stats_df) > 0 and layer_col in stats_df.columns:
            layer_stats_rows = stats_df[stats_df[layer_col] == layer]
        else:
            layer_stats_rows = None
        y_top = _annotate_saline_vs_block_brackets(ax, category_order, data_by_category, layer_stats_rows)

        ax.set_ylim(-1.3, max(1.3, y_top + 0.25))
        ax.set_yticks(np.arange(-1, 1.1, 0.5))
        ax.set_xticks(range(1, len(category_order) + 1))
        ax.set_xticklabels(category_order, rotation=45, ha='right')
        ax.set_ylabel(value_col)
        ax.set_title(layer)
        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

    for ax in axes[len(layer_order):]:
        ax.axis('off')

    fig.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    return fig

## Function -- `test_layer_depth_interaction`

Reimplemented unchanged from `5.LayerSpecific.py`'s Function 5.4 -- `rank(SMI) ~ C(condition) * C(depth_group)`. Already generic over `condition`'s categories (statsmodels just fits more dummy levels with more dcz blocks) -- no logic change needed, though with many block-levels this becomes a much larger model than the original 2-3-category version. Remember the module-level pseudo-replication caveat before reading a small interaction p-value here as confirmatory.

In [60]:
def test_layer_depth_interaction(df, layer_col='layer', group_col='condition', value_col='SMI', filter_col='valid',
                                  deep_layers=('L5', 'L6'), superficial_layers=('L2/3', 'L4')):
    """
    rank(SMI) ~ C(condition) * C(depth_group).

    Parameters
    ----------
    df : pandas.DataFrame
        One group's table.
    layer_col, group_col, value_col, filter_col : str
    deep_layers, superficial_layers : tuple of str

    Returns
    -------
    model_result : statsmodels regression results object, or None
        (fewer than 2 conditions or depth groups present).
    """
    filtered = df[df[filter_col]].copy()

    def _depth(layer):
        if layer in deep_layers:
            return 'deep'
        elif layer in superficial_layers:
            return 'superficial'
        return None

    filtered['depth_group'] = filtered[layer_col].map(_depth)
    filtered = filtered.dropna(subset=['depth_group', group_col, value_col])

    conditions_present = filtered[group_col].unique()
    depths_present = filtered['depth_group'].unique()

    if len(conditions_present) < 2 or len(depths_present) < 2:
        print(f"Not enough categories to test an interaction (conditions={list(conditions_present)}, "
              f"depths={list(depths_present)}) -- skipping.")
        return None

    filtered['SMI_rank'] = rankdata(filtered[value_col])

    formula = f"SMI_rank ~ C({group_col}) * C(depth_group)"
    model_result = smf.ols(formula, data=filtered).fit()

    print(f"\n=== Layer-depth interaction: rank({value_col}) ~ {group_col} * depth_group ===")
    print(model_result.summary().tables[1])

    interaction_terms = [p for p in model_result.params.index if ':' in p]
    if interaction_terms:
        print(f"\nInteraction term(s): {interaction_terms}")
        print("(a significant interaction term means the condition effect's SIZE differs "
              "between deep and superficial layers)")

    return model_result

## Function -- `compare_smi_by_depth_group` / `plot_smi_by_depth_group`

Reimplemented from `5.LayerSpecific.py`'s Functions 5.7/5.8 -- pools layers into `deep` (L5+L6) and `superficial` (L2/3+L4) buckets before comparing, more cells per bucket than any single layer. `compare_smi_by_depth_group` unchanged; `plot_smi_by_depth_group` uses `_condition_order_and_colors` like `plot_smi_by_layer` above.

In [61]:
def _assign_depth_group(layer_series, deep_layers=('L5', 'L6'), superficial_layers=('L2/3', 'L4')):
    """Map a 'layer' column to 'deep'/'superficial'/None."""
    def _depth(layer):
        if layer in deep_layers:
            return 'deep'
        elif layer in superficial_layers:
            return 'superficial'
        return None
    return layer_series.map(_depth)


def compare_smi_by_depth_group(df, layer_col='layer', group_col='condition', value_col='SMI', filter_col='valid',
                                deep_layers=('L5', 'L6'), superficial_layers=('L2/3', 'L4')):
    """
    Loop compare_smi_across_conditions over the two pooled depth groups
    ('deep' = L5+L6, 'superficial' = L2/3+L4) instead of all four layers.

    Parameters
    ----------
    df : pandas.DataFrame
        One group's table.
    layer_col, group_col, value_col, filter_col : str
    deep_layers, superficial_layers : tuple of str

    Returns
    -------
    results : dict
        {'deep': compare_smi_across_conditions(...) result or None,
         'superficial': ... }.
    summary_df : pandas.DataFrame
        One row per (depth_group, pairwise comparison) that had 2+ categories.
    """
    df = df.copy()
    df['depth_group'] = _assign_depth_group(df[layer_col], deep_layers, superficial_layers)

    results = {}
    summary_rows = []
    for depth_group, layers in (('deep', deep_layers), ('superficial', superficial_layers)):
        print(f"\n--- Depth group: {depth_group} ({'+'.join(layers)}) ---")
        depth_df = df[df['depth_group'] == depth_group]
        result = compare_smi_across_conditions(depth_df, group_col=group_col, value_col=value_col,
                                                filter_col=filter_col)
        results[depth_group] = result
        if result is not None:
            for _, row in result['pairwise'].iterrows():
                summary_rows.append({
                    'depth_group': depth_group, 'cond_a': row['cond_a'], 'cond_b': row['cond_b'],
                    'median_diff': row['median_diff'], 'p_holm': row['p_holm'],
                })

    summary_df = pd.DataFrame(summary_rows)
    if len(summary_df) > 0:
        print("\n=== Summary: deep vs superficial (each depth group's own Holm correction) ===")
        print(summary_df.to_string(index=False))

    return results, summary_df


def plot_smi_by_depth_group(df, layer_col='layer', group_col='condition', value_col='SMI', filter_col='valid',
                             deep_layers=('L5', 'L6'), superficial_layers=('L2/3', 'L4'), title='',
                             stats_df=None):
    """
    Violin+strip plots, one panel for 'deep' and one for 'superficial'.

    stats_df : pandas.DataFrame, optional
        compare_saline_vs_each_dcz_block_by_depth_group's output -- if
        given, draws saline-vs-each-dcz_block_N significance brackets
        (stars from Holm-corrected p, 'ns' if not significant) on each
        panel that has both 'saline' and that block.
    """
    filtered = df[df[filter_col]].copy()
    filtered['depth_group'] = _assign_depth_group(filtered[layer_col], deep_layers, superficial_layers)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    for ax, depth_group in zip(axes, ('deep', 'superficial')):
        depth_df = filtered[filtered['depth_group'] == depth_group]
        category_order, color_by_category = _condition_order_and_colors(depth_df[group_col].unique())

        data_by_category = [depth_df.loc[depth_df[group_col] == cat, value_col].to_numpy()
                            for cat in category_order]

        if len(data_by_category) == 0 or all(len(d) == 0 for d in data_by_category):
            ax.set_title(f"{depth_group} (no data)")
            ax.axis('off')
            continue

        parts = ax.violinplot(data_by_category, showmedians=True)
        for i, body in enumerate(parts['bodies']):
            body.set_facecolor(color_by_category.get(category_order[i], 'gray'))
            body.set_alpha(0.4)

        rng = np.random.default_rng(0)
        for i, vals in enumerate(data_by_category):
            jitter = rng.uniform(-0.08, 0.08, size=len(vals))
            ax.scatter(np.full(len(vals), i + 1) + jitter, vals,
                       color=color_by_category.get(category_order[i], 'gray'), s=12, alpha=0.5)

        if stats_df is not None and len(stats_df) > 0 and 'depth_group' in stats_df.columns:
            depth_stats_rows = stats_df[stats_df['depth_group'] == depth_group]
        else:
            depth_stats_rows = None
        y_top = _annotate_saline_vs_block_brackets(ax, category_order, data_by_category, depth_stats_rows)

        ax.set_ylim(-1.3, max(1.3, y_top + 0.25))
        ax.set_yticks(np.arange(-1, 1.1, 0.5))
        ax.set_xticks(range(1, len(category_order) + 1))
        ax.set_xticklabels(category_order, rotation=45, ha='right')
        ax.set_ylabel(value_col)
        ax.set_title(depth_group)
        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

    fig.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    return fig

## Function -- `summarize_layer_sample_sizes`

Reimplemented unchanged from `5.LayerSpecific.py`'s Function 5.9 -- tabulates n (valid cells) per layer x condition and per depth-group x condition, flagging anything below `low_n_threshold`. Expect MORE low-N warnings here than the original: with saline's own (small) trial count matched across every block, and now split further by layer, individual layer x block cells can get thin fast -- read this table before trusting any per-layer, per-block result above.

In [62]:
def summarize_layer_sample_sizes(df, layer_col='layer', group_col='condition', filter_col='valid',
                                  deep_layers=('L5', 'L6'), superficial_layers=('L2/3', 'L4'),
                                  low_n_threshold=10):
    """
    Tabulate n (valid cells) per layer x condition and per depth_group x
    condition, flagging anything below low_n_threshold.

    Parameters
    ----------
    df : pandas.DataFrame
        One group's table.
    layer_col, group_col, filter_col : str
    deep_layers, superficial_layers : tuple of str
    low_n_threshold : int

    Returns
    -------
    layer_counts : pandas.DataFrame
        One row per layer, one column per condition, n = valid cells.
    depth_counts : pandas.DataFrame
        Same, but for the two pooled depth groups.
    """
    filtered = df[df[filter_col]].copy()
    filtered['depth_group'] = _assign_depth_group(filtered[layer_col], deep_layers, superficial_layers)

    layers_present = [l for l in filtered[layer_col].dropna().unique()]
    layer_order = _layer_order(layers_present)
    layer_counts = filtered.groupby([layer_col, group_col]).size().unstack(fill_value=0).reindex(layer_order)

    depth_counts = filtered.groupby(['depth_group', group_col]).size().unstack(fill_value=0)
    depth_counts = depth_counts.reindex(['deep', 'superficial'])

    print("Sample sizes (valid cells) per layer x condition:")
    print(layer_counts.to_string())
    low_layer = layer_counts[layer_counts.lt(low_n_threshold).any(axis=1)]
    if len(low_layer) > 0:
        print(f"\nWARNING: layer(s) with a condition below n={low_n_threshold} "
              f"-- interpret those specific comparisons cautiously:")
        print(low_layer.to_string())

    print("\nSample sizes (valid cells) per depth group x condition:")
    print(depth_counts.to_string())
    low_depth = depth_counts[depth_counts.lt(low_n_threshold).any(axis=1)]
    if len(low_depth) > 0:
        print(f"\nWARNING: depth group(s) with a condition below n={low_n_threshold}:")
        print(low_depth.to_string())

    return layer_counts, depth_counts

## Functions -- drivers (`run_layer_analysis_for_group` / `run_layer_analysis_all_groups`)

Reimplemented unchanged from `5.LayerSpecific.py`'s Functions 5.5/5.6 -- sample-size diagnostic, then 4-layer breakdown, pooled deep/superficial comparison, and the formal interaction test, per group.

In [63]:
def run_layer_analysis_for_group(df, group_name=''):
    """
    Runs the sample-size diagnostic, 4-layer breakdown, pooled
    deep/superficial comparison, and interaction test for one group.
    Retains both figures under 'layer_fig'/'depth_fig' so the save step
    can save them without needing to replot. Both figures are annotated
    with saline-vs-each-dcz_block_N significance brackets (from
    compare_saline_vs_each_dcz_block_by_layer/_by_depth_group, called
    here with verbose=False since the dedicated saline-vs-blocks driver
    step below prints/saves those same tables already).

    Parameters
    ----------
    df : pandas.DataFrame
        One group's table.
    group_name : str

    Returns
    -------
    result : dict with keys: layer_counts, depth_counts, layer_results,
        layer_summary, layer_fig, depth_results, depth_summary, depth_fig,
        interaction_result.
    """
    print(f"\n{'='*90}\nLayer analysis: {group_name}\n{'='*90}")

    layer_counts, depth_counts = summarize_layer_sample_sizes(df)

    layer_results, layer_summary_df = compare_smi_by_layer(df)
    layer_block_stats_df = compare_saline_vs_each_dcz_block_by_layer(df, verbose=False)
    layer_fig = plot_smi_by_layer(df, title=group_name, stats_df=layer_block_stats_df)
    plt.close(layer_fig)

    depth_results, depth_summary_df = compare_smi_by_depth_group(df)
    depth_block_stats_df = compare_saline_vs_each_dcz_block_by_depth_group(df, verbose=False)
    depth_fig = plot_smi_by_depth_group(df, title=f"{group_name} (deep vs superficial)",
                                         stats_df=depth_block_stats_df)
    plt.close(depth_fig)

    interaction_result = test_layer_depth_interaction(df)

    return {
        'layer_counts': layer_counts,
        'depth_counts': depth_counts,
        'layer_results': layer_results,
        'layer_summary': layer_summary_df,
        'layer_fig': layer_fig,
        'depth_results': depth_results,
        'depth_summary': depth_summary_df,
        'depth_fig': depth_fig,
        'interaction_result': interaction_result,
    }


def run_layer_analysis_all_groups(all_group_dfs, group_col='condition', filter_col='valid'):
    """
    Loops run_layer_analysis_for_group over every group, skipping
    single-condition groups (shouldn't normally happen here -- every
    trial-matched group has at least saline + dcz_block_1 -- kept as a
    defensive check anyway).

    Parameters
    ----------
    all_group_dfs : dict
        {group_name: df}, from load_all_group_dfs_from_trial_matched.
    group_col, filter_col : str

    Returns
    -------
    results : dict
        {group_name: run_layer_analysis_for_group(...) result}.
    """
    results = {}
    for group_name, df in all_group_dfs.items():
        conditions_present = df.loc[df[filter_col], group_col].unique()
        if len(conditions_present) < 2:
            print(f"\n{'='*90}\n{group_name}: only {len(conditions_present)} condition(s) present "
                  f"({list(conditions_present)}) -- skipping layer analysis for this group.\n{'='*90}")
            continue
        results[group_name] = run_layer_analysis_for_group(df, group_name=group_name)

    return results

## Functions -- save everything

Reimplemented unchanged in spirit from `5.LayerSpecific.py`'s Function 5.10 -- sample-size tables, per-layer and per-depth-group summary stats (+ omnibus JSON), both figures, and the interaction regression's text summary, per group. `layer_fig`/`depth_fig` are already `plt.close`d by `run_layer_analysis_for_group` above (no popup windows in the driver path, same convention `4.SessionComparison_TrialMatched.ipynb` uses) -- saved from the figure object regardless of whether it was ever shown.

In [64]:
def save_dataframe_csv(df, output_dir, filename, index=False):
    """
    Save a DataFrame to {output_dir}/{filename}, creating output_dir if
    needed. index=True for tables whose index is meaningful (e.g. layer
    names), False for tables with a plain range index.
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    df.to_csv(save_path, index=index)
    print(f"Saved -> {save_path}")
    return save_path


def save_figure_png(fig, output_dir, filename, dpi=150):
    """
    Save a matplotlib figure to {output_dir}/{filename}, creating
    output_dir if needed.
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    print(f"Saved -> {save_path}")
    return save_path


def _json_safe(obj):
    """Recursively convert numpy scalar types to native Python for json.dump."""
    if isinstance(obj, dict):
        return {k: _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj


def save_json(data, output_dir, filename):
    """
    Save a JSON-serializable dict to {output_dir}/{filename}, creating
    output_dir if needed.
    """
    import json
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    with open(save_path, 'w') as f:
        json.dump(_json_safe(data), f, indent=2)
    print(f"Saved -> {save_path}")
    return save_path


def _extract_omnibus_summary(results_by_category):
    """
    Pull {category: {'omnibus_stat', 'omnibus_p', 'group_medians', 'group_n'}}
    out of a {category: compare_smi_across_conditions result or None} dict,
    for JSON saving (skips categories where result was None).
    """
    summary = {}
    for cat, result in results_by_category.items():
        if result is None:
            continue
        summary[cat] = {
            'omnibus_stat': result['omnibus_stat'],
            'omnibus_p': result['omnibus_p'],
            'group_medians': result['group_medians'],
            'group_n': result['group_n'],
        }
    return summary


def save_layer_analysis_group_outputs(output_dir, group_name, result):
    """
    Save one group's layer-analysis outputs: sample-size tables, per-layer
    and per-depth-group summary stats (+ omnibus JSON), both figures, and
    the interaction regression's text summary.

    Parameters
    ----------
    output_dir : str
    group_name : str
    result : dict
        From run_layer_analysis_for_group.

    Returns
    -------
    saved_paths : dict
    """
    saved_paths = {
        'layer_sample_sizes': save_dataframe_csv(
            result['layer_counts'], output_dir, f"{group_name}_layer_sample_sizes.csv", index=True),
        'depth_sample_sizes': save_dataframe_csv(
            result['depth_counts'], output_dir, f"{group_name}_depth_sample_sizes.csv", index=True),
    }

    if len(result['layer_summary']) > 0:
        saved_paths['layer_pairwise'] = save_dataframe_csv(
            result['layer_summary'], output_dir, f"{group_name}_layer_pairwise_stats.csv")
    if len(result['depth_summary']) > 0:
        saved_paths['depth_pairwise'] = save_dataframe_csv(
            result['depth_summary'], output_dir, f"{group_name}_depth_pairwise_stats.csv")

    saved_paths['layer_omnibus'] = save_json(
        _extract_omnibus_summary(result['layer_results']), output_dir, f"{group_name}_layer_omnibus_stats.json")
    saved_paths['depth_omnibus'] = save_json(
        _extract_omnibus_summary(result['depth_results']), output_dir, f"{group_name}_depth_omnibus_stats.json")

    layer_fig = result.get('layer_fig')
    if layer_fig is not None:
        saved_paths['layer_plot'] = save_figure_png(layer_fig, output_dir, f"{group_name}_layer_comparison_plot.png")

    depth_fig = result.get('depth_fig')
    if depth_fig is not None:
        saved_paths['depth_plot'] = save_figure_png(depth_fig, output_dir, f"{group_name}_depth_comparison_plot.png")

    interaction_result = result.get('interaction_result')
    if interaction_result is not None:
        os.makedirs(output_dir, exist_ok=True)
        txt_path = os.path.join(output_dir, f"{group_name}_interaction_regression.txt")
        with open(txt_path, 'w') as f:
            f.write(str(interaction_result.summary()))
        print(f"Saved -> {txt_path}")
        saved_paths['interaction_regression'] = txt_path

    return saved_paths


def save_all_layer_analysis_outputs(output_dir, layer_analysis_results):
    """
    Loop save_layer_analysis_group_outputs over every group.

    Parameters
    ----------
    output_dir : str
    layer_analysis_results : dict
        {group_name: run_layer_analysis_for_group(...) result}.

    Returns
    -------
    saved_paths_by_group : dict
    """
    saved_paths_by_group = {}
    for group_name, result in layer_analysis_results.items():
        saved_paths_by_group[group_name] = save_layer_analysis_group_outputs(output_dir, group_name, result)
    print(f"\nSaved layer-analysis outputs for {len(saved_paths_by_group)} group(s) to {output_dir}")
    return saved_paths_by_group

## Functions -- `compare_saline_vs_each_dcz_block` (whole population / by layer / by depth group)

`compare_smi_across_conditions`'s pairwise table above already contains every saline-vs-block comparison -- but buried among all the block-vs-block ones too, with Holm correction applied across that whole larger combinatorial set. These functions isolate just the comparisons you actually asked for -- saline vs. `dcz_block_1`, saline vs. `dcz_block_2`, ..., saline vs. `dcz_block_n` -- and Holm-correct only across that specific family (n_blocks tests), not the larger set, at three levels: whole population, per layer, and per pooled depth group. `run_saline_vs_blocks_analysis_all_groups` (below, wired into the final driver) runs all three for every comparison group -- "all dcz sessions" -- and consolidates each level into one table across groups.

In [65]:
def _compare_saline_vs_dcz_blocks_core(df, group_col='condition', value_col='SMI', filter_col='valid'):
    """
    Mann-Whitney U, saline vs. each dcz_block_N present (NOT block-vs-
    block) -- Holm-corrected only across this specific family of tests.

    Parameters
    ----------
    df : pandas.DataFrame
        Already whatever subset you want tested (whole population, one
        layer, one depth group, ...).
    group_col, value_col, filter_col : str

    Returns
    -------
    summary_df : pandas.DataFrame
        One row per dcz_block_N present: median_saline, median_block,
        median_diff, n_saline, n_block, U_stat, p_raw, p_holm. Empty if
        'saline' or no 'dcz_block_N' is present in df.
    """
    filtered = df[df[filter_col]]
    conditions_present = [c for c in filtered[group_col].unique() if pd.notna(c)]

    if 'saline' not in conditions_present:
        return pd.DataFrame()

    dcz_blocks = sorted(
        [c for c in conditions_present if str(c).startswith('dcz_block_')],
        key=lambda c: int(str(c).rsplit('_', 1)[1])
    )
    if not dcz_blocks:
        return pd.DataFrame()

    saline_vals = filtered.loc[filtered[group_col] == 'saline', value_col].to_numpy()

    rows = []
    for block in dcz_blocks:
        block_vals = filtered.loc[filtered[group_col] == block, value_col].to_numpy()
        if len(saline_vals) == 0 or len(block_vals) == 0:
            continue
        u_stat, p_raw = mannwhitneyu(saline_vals, block_vals, alternative='two-sided')
        rows.append({
            'dcz_block': block,
            'median_saline': float(np.median(saline_vals)),
            'median_block': float(np.median(block_vals)),
            'median_diff': float(np.median(saline_vals) - np.median(block_vals)),
            'n_saline': len(saline_vals), 'n_block': len(block_vals),
            'U_stat': u_stat, 'p_raw': p_raw,
        })

    summary_df = pd.DataFrame(rows)
    if len(summary_df) > 0:
        _, p_holm, _, _ = multipletests(summary_df['p_raw'], method='holm')
        summary_df['p_holm'] = p_holm

    return summary_df


def compare_saline_vs_each_dcz_block(df, group_col='condition', value_col='SMI', filter_col='valid', verbose=True):
    """
    Whole-population version: saline vs. each dcz_block_N, Holm-corrected
    only across these comparisons.

    Parameters
    ----------
    df : pandas.DataFrame
        One group's table.
    group_col, value_col, filter_col : str
    verbose : bool
        Print the result table. Set False when calling this just to get
        stats_df for a plot annotation (avoids duplicate console output
        when the dedicated saline-vs-blocks driver step prints it again).

    Returns
    -------
    summary_df : pandas.DataFrame
    """
    summary_df = _compare_saline_vs_dcz_blocks_core(df, group_col=group_col, value_col=value_col,
                                                     filter_col=filter_col)
    if verbose:
        if len(summary_df) > 0:
            print("Saline vs. each dcz block (whole population, Holm-corrected across blocks only):")
            print(summary_df.to_string(index=False))
        else:
            print("Not enough categories present (need 'saline' + at least one 'dcz_block_N').")
    return summary_df


def compare_saline_vs_each_dcz_block_by_layer(df, layer_col='layer', group_col='condition',
                                               value_col='SMI', filter_col='valid', verbose=True):
    """
    Loop _compare_saline_vs_dcz_blocks_core over each layer present.

    Parameters
    ----------
    df : pandas.DataFrame
        One group's table.
    layer_col, group_col, value_col, filter_col : str
    verbose : bool
        Print the result table (see compare_saline_vs_each_dcz_block).

    Returns
    -------
    result_df : pandas.DataFrame
        One row per (layer, dcz_block).
    """
    layers_present = [l for l in df[layer_col].dropna().unique()]
    layer_order = _layer_order(layers_present)

    layer_dfs = []
    for layer in layer_order:
        layer_df = df[df[layer_col] == layer]
        summary_df = _compare_saline_vs_dcz_blocks_core(layer_df, group_col=group_col, value_col=value_col,
                                                         filter_col=filter_col)
        if len(summary_df) > 0:
            summary_df.insert(0, 'layer', layer)
            layer_dfs.append(summary_df)

    result_df = pd.concat(layer_dfs, ignore_index=True) if layer_dfs else pd.DataFrame()
    if verbose and len(result_df) > 0:
        print("Saline vs. each dcz block, by layer (each layer's own Holm correction across blocks only):")
        print(result_df.to_string(index=False))
    return result_df


def compare_saline_vs_each_dcz_block_by_depth_group(df, layer_col='layer', group_col='condition',
                                                     value_col='SMI', filter_col='valid',
                                                     deep_layers=('L5', 'L6'), superficial_layers=('L2/3', 'L4'),
                                                     verbose=True):
    """
    Loop _compare_saline_vs_dcz_blocks_core over the two pooled depth
    groups.

    Parameters
    ----------
    df : pandas.DataFrame
        One group's table.
    layer_col, group_col, value_col, filter_col : str
    deep_layers, superficial_layers : tuple of str
    verbose : bool
        Print the result table (see compare_saline_vs_each_dcz_block).

    Returns
    -------
    result_df : pandas.DataFrame
        One row per (depth_group, dcz_block).
    """
    df = df.copy()
    df['depth_group'] = _assign_depth_group(df[layer_col], deep_layers, superficial_layers)

    depth_dfs = []
    for depth_group in ('deep', 'superficial'):
        depth_df = df[df['depth_group'] == depth_group]
        summary_df = _compare_saline_vs_dcz_blocks_core(depth_df, group_col=group_col, value_col=value_col,
                                                         filter_col=filter_col)
        if len(summary_df) > 0:
            summary_df.insert(0, 'depth_group', depth_group)
            depth_dfs.append(summary_df)

    result_df = pd.concat(depth_dfs, ignore_index=True) if depth_dfs else pd.DataFrame()
    if verbose and len(result_df) > 0:
        print("Saline vs. each dcz block, by depth group (each depth group's own Holm correction across blocks only):")
        print(result_df.to_string(index=False))
    return result_df

In [66]:
def run_saline_vs_blocks_analysis_all_groups(all_group_dfs):
    """
    Loop compare_saline_vs_each_dcz_block(_by_layer/_by_depth_group) over
    every comparison group ("all dcz sessions"), tagging each result with
    its group name, and concatenate into three consolidated tables.

    Parameters
    ----------
    all_group_dfs : dict
        {group_name: df}, from load_all_group_dfs_from_trial_matched.

    Returns
    -------
    whole_pop_df, layer_df, depth_df : pandas.DataFrame
        Each with a leading 'group' column, one row per (group, ..., dcz_block).
    """
    whole_pop_rows, layer_rows, depth_rows = [], [], []

    for group_name, df in all_group_dfs.items():
        print(f"\n{'='*90}\n{group_name}\n{'='*90}")

        whole_pop_result = compare_saline_vs_each_dcz_block(df)
        if len(whole_pop_result) > 0:
            whole_pop_result.insert(0, 'group', group_name)
            whole_pop_rows.append(whole_pop_result)

        layer_result = compare_saline_vs_each_dcz_block_by_layer(df)
        if len(layer_result) > 0:
            layer_result.insert(0, 'group', group_name)
            layer_rows.append(layer_result)

        depth_result = compare_saline_vs_each_dcz_block_by_depth_group(df)
        if len(depth_result) > 0:
            depth_result.insert(0, 'group', group_name)
            depth_rows.append(depth_result)

    whole_pop_df = pd.concat(whole_pop_rows, ignore_index=True) if whole_pop_rows else pd.DataFrame()
    layer_df = pd.concat(layer_rows, ignore_index=True) if layer_rows else pd.DataFrame()
    depth_df = pd.concat(depth_rows, ignore_index=True) if depth_rows else pd.DataFrame()

    print(f"\n{'='*90}\nConsolidated across all {len(all_group_dfs)} group(s)\n{'='*90}")
    print("\nWhole population:")
    print(whole_pop_df.to_string(index=False) if len(whole_pop_df) else "(none)")
    print("\nBy layer:")
    print(layer_df.to_string(index=False) if len(layer_df) else "(none)")
    print("\nBy depth group:")
    print(depth_df.to_string(index=False) if len(depth_df) else "(none)")

    return whole_pop_df, layer_df, depth_df


def save_saline_vs_blocks_outputs(output_dir, whole_pop_df, layer_df, depth_df):
    """
    Save the three consolidated saline-vs-each-dcz-block tables.

    Parameters
    ----------
    output_dir : str
    whole_pop_df, layer_df, depth_df : pandas.DataFrame
        From run_saline_vs_blocks_analysis_all_groups.

    Returns
    -------
    saved_paths : dict
    """
    saved_paths = {}
    if len(whole_pop_df) > 0:
        saved_paths['whole_population'] = save_dataframe_csv(
            whole_pop_df, output_dir, 'saline_vs_dcz_blocks_whole_population.csv')
    if len(layer_df) > 0:
        saved_paths['by_layer'] = save_dataframe_csv(
            layer_df, output_dir, 'saline_vs_dcz_blocks_by_layer.csv')
    if len(depth_df) > 0:
        saved_paths['by_depth_group'] = save_dataframe_csv(
            depth_df, output_dir, 'saline_vs_dcz_blocks_by_depth_group.csv')
    return saved_paths

## Functions -- corrected paired test across independent comparison GROUPS, per dcz block (`5.LayerSpecific.py`'s Functions 5.11-5.14, adapted)

Everything above -- including `compare_saline_vs_each_dcz_block*` -- pools cells within one group as independent samples, when for any one group there's really only one saline session and one paired dcz session (the pseudo-replication caveat flagged at the top of this notebook). `5.LayerSpecific.py` fixed this for its own single `'dcz'` condition with Functions 5.11-5.14: collapse each comparison group down to one median-SMI number per layer, then treat the comparison GROUPS themselves (n=5: DCZ1/2/3, Active_OL, Stationary_OL) as the independent paired samples for a paired t-test (+ Wilcoxon) and a repeated-measures ANOVA across layers.

These functions are that same fix, just repeated once per `dcz_block_N` position instead of once for a single `'dcz'` condition -- `dcz_block_1` is compared across whichever groups have a `dcz_block_1`, `dcz_block_2` across whichever have a `dcz_block_2`, etc. (groups with fewer blocks contribute to fewer of these). Not Holm-corrected across layers/blocks, matching the original's own choice not to correct Function 5.12 across layers -- only the layer-vs-layer heterogeneity test (Function 5.13's analog below) Holm-corrects, and only within one block's own layer-pair family.</cell id="d973d95c">


In [ ]:
from scipy.stats import ttest_rel, wilcoxon


def compute_per_group_layer_smi_summary_by_block(all_group_dfs, layer_col='layer', group_col='condition',
                                                  value_col='SMI', filter_col='valid'):
    """
    Per comparison group x layer x dcz_block_N: median SMI for saline and
    that block's cells (valid-filtered), plus n. Adapted from
    5.LayerSpecific.py's compute_per_group_layer_smi_summary (Function
    5.11) for trial-matched data's dcz_block_N conditions instead of a
    single 'dcz' -- each block position gets its own row per (group,
    layer), so test_paired_smi_significance_by_block_and_layer below can
    pair across the independent comparison GROUPS separately for each
    block position.

    Also includes 'within_group_mannwhitney_p' -- same caveat as the
    original: descriptive only (cells within one recording are
    pseudo-replicated), here to show which recording drives the paired
    test below, not to replace it.

    Parameters
    ----------
    all_group_dfs : dict
        {group_name: df}, from load_all_group_dfs_from_trial_matched.
    layer_col, group_col, value_col, filter_col : str

    Returns
    -------
    summary_df : pandas.DataFrame
        One row per (group, layer, dcz_block) where both saline and that
        block are present: group, layer, dcz_block, saline_median,
        block_median, diff, n_saline, n_block, within_group_mannwhitney_p.
    """
    rows = []
    for group_name, df in all_group_dfs.items():
        filtered = df[df[filter_col]]
        conditions_present = [c for c in filtered[group_col].unique() if pd.notna(c)]
        if 'saline' not in conditions_present:
            continue

        dcz_blocks = sorted(
            [c for c in conditions_present if str(c).startswith('dcz_block_')],
            key=lambda c: int(str(c).rsplit('_', 1)[1])
        )
        if not dcz_blocks:
            continue

        layers_present = [l for l in filtered[layer_col].dropna().unique()]
        for layer in _layer_order(layers_present):
            layer_df = filtered[filtered[layer_col] == layer]
            saline_vals = layer_df.loc[layer_df[group_col] == 'saline', value_col].to_numpy()
            if len(saline_vals) == 0:
                continue
            saline_median = float(np.median(saline_vals))

            for block in dcz_blocks:
                block_vals = layer_df.loc[layer_df[group_col] == block, value_col].to_numpy()
                if len(block_vals) == 0:
                    continue
                block_median = float(np.median(block_vals))

                try:
                    _, p_within_group = mannwhitneyu(saline_vals, block_vals, alternative='two-sided')
                except ValueError:
                    p_within_group = np.nan

                rows.append({
                    'group': group_name, 'layer': layer, 'dcz_block': block,
                    'saline_median': saline_median, 'block_median': block_median,
                    'diff': saline_median - block_median,
                    'n_saline': len(saline_vals), 'n_block': len(block_vals),
                    'within_group_mannwhitney_p': p_within_group,
                })

    summary_df = pd.DataFrame(rows)
    print(summary_df.to_string(index=False))
    print("\nNOTE: 'within_group_mannwhitney_p' treats cells within one recording as independent")
    print("samples (pseudo-replicated, same issue as compare_smi_by_layer above) -- descriptive,")
    print("not a reliable standalone significance test the way the paired t-test below is.")
    return summary_df


def test_paired_smi_significance_by_block_and_layer(summary_df):
    """
    Paired t-test (+ Wilcoxon, for comparison) on median SMI, saline vs.
    each dcz_block_N, per (dcz_block, layer) combination, across the
    independent comparison groups that have that block position present.
    Adapted from 5.LayerSpecific.py's test_paired_smi_significance_by_layer
    (Function 5.12) -- not Holm-corrected across layers/blocks here
    either, matching the original (its per-layer tests weren't corrected
    against each other; only the layer-vs-layer heterogeneity test below
    Holm-corrects, same as the original's Function 5.13).

    Parameters
    ----------
    summary_df : pandas.DataFrame
        From compute_per_group_layer_smi_summary_by_block.

    Returns
    -------
    result_df : pandas.DataFrame
        One row per (dcz_block, layer): n_pairs, mean_diff, std_diff,
        n_same_direction, paired_t_stat, paired_t_p, wilcoxon_p. Skips
        (block, layer) combinations with fewer than 2 groups present
        (can't paired-test with <2 pairs).
    """
    if len(summary_df) == 0:
        print("No (group, layer, dcz_block) rows to test.")
        return pd.DataFrame()

    dcz_blocks = sorted(summary_df['dcz_block'].unique(), key=lambda c: int(str(c).rsplit('_', 1)[1]))

    rows = []
    for block in dcz_blocks:
        block_rows_all = summary_df[summary_df['dcz_block'] == block]
        for layer in _layer_order(block_rows_all['layer'].unique()):
            layer_rows = block_rows_all[block_rows_all['layer'] == layer]
            saline_vals = layer_rows['saline_median'].to_numpy()
            block_vals = layer_rows['block_median'].to_numpy()
            n_pairs = len(saline_vals)

            if n_pairs < 2:
                print(f"{block} / {layer}: only {n_pairs} pair(s) -- skipping (need >=2 for a paired test).")
                continue

            diffs = saline_vals - block_vals
            t_stat, p_ttest = ttest_rel(saline_vals, block_vals)
            if np.all(diffs == diffs[0]):
                w_stat, p_wilcoxon = np.nan, np.nan
            else:
                w_stat, p_wilcoxon = wilcoxon(saline_vals, block_vals)

            rows.append({
                'dcz_block': block, 'layer': layer, 'n_pairs': n_pairs,
                'mean_diff': diffs.mean(), 'std_diff': diffs.std(ddof=1) if n_pairs > 1 else np.nan,
                'n_same_direction': int((diffs > 0).sum()),
                'paired_t_stat': t_stat, 'paired_t_p': p_ttest, 'wilcoxon_p': p_wilcoxon,
            })

    result_df = pd.DataFrame(rows)
    print(result_df.to_string(index=False))
    return result_df

In [ ]:
def test_layer_heterogeneity_in_smi_diff_by_block(summary_df, diff_col='diff'):
    """
    For each dcz_block_N separately: does the SIZE of the saline-block
    SMI difference vary significantly across layers? Repeated-measures
    ANOVA (subject = group, within = layer) as the omnibus test, then
    Holm-corrected pairwise paired t-tests between layers -- adapted from
    5.LayerSpecific.py's test_layer_heterogeneity_in_smi_diff (Function
    5.13), just run once per block position instead of once overall.

    Only uses groups that have all layers present for that block
    (AnovaRM needs balanced data) -- prints a warning listing any group
    dropped for this reason, per block.

    Parameters
    ----------
    summary_df : pandas.DataFrame
        From compute_per_group_layer_smi_summary_by_block.
    diff_col : str

    Returns
    -------
    aov_results_by_block : dict
        {dcz_block: statsmodels AnovaRM results object}, only for blocks
        with >=2 complete groups.
    pairwise_df : pandas.DataFrame
        One row per (dcz_block, layer_a, layer_b): mean_diff_a,
        mean_diff_b, t_stat, p_raw, p_holm (Holm-corrected within each
        block's own layer-pair family, not across blocks).
    """
    from statsmodels.stats.anova import AnovaRM

    if len(summary_df) == 0:
        print("No (group, layer, dcz_block) rows to test.")
        return {}, pd.DataFrame()

    dcz_blocks = sorted(summary_df['dcz_block'].unique(), key=lambda c: int(str(c).rsplit('_', 1)[1]))

    aov_results_by_block = {}
    all_pairwise_rows = []

    for block in dcz_blocks:
        block_df = summary_df[summary_df['dcz_block'] == block]
        pivot = block_df.pivot(index='group', columns='layer', values=diff_col)
        complete = pivot.dropna()
        dropped = set(pivot.index) - set(complete.index)
        if dropped:
            print(f"{block}: dropped {len(dropped)} group(s) missing one or more layers "
                  f"(AnovaRM needs balanced data): {sorted(dropped)}")

        if len(complete) < 2:
            print(f"{block}: only {len(complete)} complete group(s) -- skipping (need >=2).")
            continue

        rm_df = complete.reset_index().melt(id_vars='group', var_name='layer', value_name=diff_col)
        aov_result = AnovaRM(rm_df, depvar=diff_col, subject='group', within=['layer']).fit()
        aov_results_by_block[block] = aov_result
        print(f"\n=== {block}: Repeated-measures ANOVA -- does the saline-block {diff_col} vary by layer? ===")
        print(aov_result)

        layers = list(complete.columns)
        pairwise_rows = []
        for layer_a, layer_b in combinations(layers, 2):
            t_stat, p_raw = ttest_rel(complete[layer_a], complete[layer_b])
            pairwise_rows.append({
                'dcz_block': block, 'layer_a': layer_a, 'layer_b': layer_b,
                'mean_diff_a': complete[layer_a].mean(), 'mean_diff_b': complete[layer_b].mean(),
                't_stat': t_stat, 'p_raw': p_raw,
            })
        pairwise_block_df = pd.DataFrame(pairwise_rows)
        if len(pairwise_block_df) > 0:
            _, p_holm, _, _ = multipletests(pairwise_block_df['p_raw'], method='holm')
            pairwise_block_df['p_holm'] = p_holm

        print(f"\n{block}: pairwise layer-vs-layer comparison of the saline-block {diff_col} "
              f"(Holm-corrected within this block only):")
        print(pairwise_block_df.to_string(index=False))
        all_pairwise_rows.append(pairwise_block_df)

    pairwise_df = pd.concat(all_pairwise_rows, ignore_index=True) if all_pairwise_rows else pd.DataFrame()
    return aov_results_by_block, pairwise_df


In [ ]:
def _group_color_map(groups):
    """
    Assign each comparison group name a distinct, stable color. The
    original 5.LayerSpecific.py's GROUP_COLORS dict is keyed by short
    labels ('DCZ1', 'Active_OL', ...) that don't match this notebook's
    actual group names (full session-folder names from
    load_all_group_dfs_from_trial_matched) -- so build a fresh
    tab10/tab20 categorical map instead, sorted for a stable order run
    to run.
    """
    groups = sorted(groups)
    cmap = plt.cm.tab10 if len(groups) <= 10 else plt.cm.tab20
    return {g: cmap(i % cmap.N) for i, g in enumerate(groups)}


def plot_paired_smi_by_layer_by_block(summary_df, dcz_block, title=''):
    """
    Small multiples (one panel per layer): one line per comparison group,
    saline -> this specific dcz_block_N's median SMI. Same slope-plot
    design as 5.LayerSpecific.py's plot_paired_smi_by_layer (Function
    5.14), filtered to one block position at a time, with
    _group_color_map instead of the original's hardcoded GROUP_COLORS.

    Parameters
    ----------
    summary_df : pandas.DataFrame
        From compute_per_group_layer_smi_summary_by_block.
    dcz_block : str
        Which block position to plot (e.g. 'dcz_block_1').
    title : str

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    block_df = summary_df[summary_df['dcz_block'] == dcz_block]
    layer_order = _layer_order(block_df['layer'].unique())
    groups = sorted(block_df['group'].unique())
    group_colors = _group_color_map(groups)

    fig, axes = plt.subplots(1, len(layer_order), figsize=(5 * len(layer_order), 7.5), sharey=True)
    axes = np.atleast_1d(axes)
    x_positions = {'saline': 0, 'block': 1}

    for ax, layer in zip(axes, layer_order):
        layer_rows = block_df[block_df['layer'] == layer]
        for _, row in layer_rows.iterrows():
            color = group_colors.get(row['group'], 'gray')
            ax.plot([x_positions['saline'], x_positions['block']],
                    [row['saline_median'], row['block_median']],
                    color=color, marker='o', markersize=8, linewidth=2,
                    solid_capstyle='round', zorder=3)

        ax.set_xlim(-0.3, 1.3)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['saline', dcz_block])
        ax.axhline(0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
        ax.set_title(layer)

    axes[0].set_ylabel('Median SMI')

    handles = [plt.Line2D([0], [0], color=group_colors.get(g, 'gray'), marker='o', linewidth=2, label=g)
              for g in groups]
    fig.legend(handles=handles, loc='lower center', ncol=min(len(handles), 3), fontsize=9,
              bbox_to_anchor=(0.5, -0.12), frameon=False)

    fig.suptitle(title if title else f'Median SMI, saline -> {dcz_block}, by layer (paired by comparison group)')
    plt.tight_layout()
    return fig


def run_paired_layer_smi_analysis_by_block(all_group_dfs):
    """
    Runs the corrected (n=independent comparison groups, not n=pooled
    cells) version of the saline-vs-each-dcz-block layer question --
    adapted from 5.LayerSpecific.py's Functions 5.11-5.14
    (run_paired_layer_smi_analysis), just repeated once per dcz_block_N
    position instead of once for a single 'dcz' condition.

    Parameters
    ----------
    all_group_dfs : dict
        {group_name: df}, from load_all_group_dfs_from_trial_matched.

    Returns
    -------
    summary_df : pandas.DataFrame
    block_layer_significance_df : pandas.DataFrame
    aov_results_by_block : dict
    layer_heterogeneity_df : pandas.DataFrame
    figs_by_block : dict
        {dcz_block: matplotlib.figure.Figure}, already plt.close()'d (no
        popup windows, same convention as the rest of this notebook).
    """
    print(f"\n{'='*90}\nCorrected paired-significance layer SMI analysis, by dcz block\n{'='*90}")

    summary_df = compute_per_group_layer_smi_summary_by_block(all_group_dfs)
    block_layer_significance_df = test_paired_smi_significance_by_block_and_layer(summary_df)
    aov_results_by_block, layer_heterogeneity_df = test_layer_heterogeneity_in_smi_diff_by_block(summary_df)

    figs_by_block = {}
    if len(summary_df) > 0:
        dcz_blocks = sorted(summary_df['dcz_block'].unique(), key=lambda c: int(str(c).rsplit('_', 1)[1]))
        for block in dcz_blocks:
            fig = plot_paired_smi_by_layer_by_block(summary_df, block)
            plt.close(fig)
            figs_by_block[block] = fig

    return summary_df, block_layer_significance_df, aov_results_by_block, layer_heterogeneity_df, figs_by_block


In [ ]:
def save_paired_layer_smi_by_block_outputs(output_dir, summary_df, block_layer_significance_df,
                                            aov_results_by_block, layer_heterogeneity_df, figs_by_block):
    """
    Saves run_paired_layer_smi_analysis_by_block's outputs:
    paired_layer_smi_by_block_summary.csv,
    paired_layer_smi_by_block_significance.csv,
    layer_heterogeneity_by_block_pairwise.csv, one slope-plot PNG and one
    AnovaRM text summary per dcz_block. Adapted from
    5.LayerSpecific.py's save_paired_layer_smi_outputs.

    Parameters
    ----------
    output_dir : str
    summary_df, block_layer_significance_df, layer_heterogeneity_df : pandas.DataFrame
    aov_results_by_block, figs_by_block : dict
        {dcz_block: ...}, from run_paired_layer_smi_analysis_by_block.

    Returns
    -------
    saved_paths : dict
    """
    saved_paths = {}
    if len(summary_df) > 0:
        saved_paths['summary'] = save_dataframe_csv(
            summary_df, output_dir, 'paired_layer_smi_by_block_summary.csv')
    if len(block_layer_significance_df) > 0:
        saved_paths['significance'] = save_dataframe_csv(
            block_layer_significance_df, output_dir, 'paired_layer_smi_by_block_significance.csv')
    if len(layer_heterogeneity_df) > 0:
        saved_paths['heterogeneity'] = save_dataframe_csv(
            layer_heterogeneity_df, output_dir, 'layer_heterogeneity_by_block_pairwise.csv')

    for block, fig in figs_by_block.items():
        saved_paths[f'{block}_plot'] = save_figure_png(
            fig, output_dir, f'paired_layer_smi_slope_{block}.png')

    for block, aov_result in aov_results_by_block.items():
        os.makedirs(output_dir, exist_ok=True)
        txt_path = os.path.join(output_dir, f'layer_heterogeneity_anova_{block}.txt')
        with open(txt_path, 'w') as f:
            f.write(str(aov_result))
        print(f"Saved -> {txt_path}")
        saved_paths[f'{block}_anova'] = txt_path

    return saved_paths


## Run it -- both animals, no popup windows

Same no-blocking, dual-animal convention `4.SessionComparison_TrialMatched.ipynb`'s final driver cell uses: every figure is created, saved, and closed without ever calling `plt.show()`, so this runs start to finish unattended for both JSY090 and JSY093. Review results afterward from the saved PNGs/CSVs, or call `plt.show()` yourself on a returned `fig` if you want to look at one interactively.

In [ ]:
ANIMAL_CONFIGS = [
    {'animal_label': 'JSY090', 'animal_dir': TEST_ANIMAL_DIR_JSY090},
    {'animal_label': 'JSY093', 'animal_dir': TEST_ANIMAL_DIR_JSY093},
]

layer_trial_matched_results_by_animal = {}
for animal_cfg in ANIMAL_CONFIGS:
    print(f"\n{'#'*90}\n{animal_cfg['animal_label']}\n{'#'*90}")
    phase4_trial_matched_dir = os.path.join(animal_cfg['animal_dir'], 'Phase4_TrialMatched_Results')
    phase5_output_dir = os.path.join(animal_cfg['animal_dir'], 'Phase5_LayerSpecific_TrialMatched_Results')

    all_group_dfs = load_all_group_dfs_from_trial_matched(phase4_trial_matched_dir)
    layer_analysis_results = run_layer_analysis_all_groups(all_group_dfs)
    saved_paths_by_group = save_all_layer_analysis_outputs(phase5_output_dir, layer_analysis_results)

    whole_pop_df, layer_df, depth_df = run_saline_vs_blocks_analysis_all_groups(all_group_dfs)
    saline_vs_blocks_saved_paths = save_saline_vs_blocks_outputs(phase5_output_dir, whole_pop_df, layer_df, depth_df)

    (paired_by_block_summary_df, paired_by_block_significance_df, paired_by_block_aov_results,
     paired_by_block_heterogeneity_df, paired_by_block_figs) = run_paired_layer_smi_analysis_by_block(all_group_dfs)
    paired_by_block_saved_paths = save_paired_layer_smi_by_block_outputs(
        phase5_output_dir, paired_by_block_summary_df, paired_by_block_significance_df,
        paired_by_block_aov_results, paired_by_block_heterogeneity_df, paired_by_block_figs)

    layer_trial_matched_results_by_animal[animal_cfg['animal_label']] = {
        'all_group_dfs': all_group_dfs,
        'layer_analysis_results': layer_analysis_results,
        'saved_paths_by_group': saved_paths_by_group,
        'saline_vs_blocks': {'whole_population': whole_pop_df, 'by_layer': layer_df, 'by_depth_group': depth_df},
        'saline_vs_blocks_saved_paths': saline_vs_blocks_saved_paths,
        'paired_by_block': {
            'summary': paired_by_block_summary_df,
            'significance': paired_by_block_significance_df,
            'aov_results': paired_by_block_aov_results,
            'heterogeneity': paired_by_block_heterogeneity_df,
        },
        'paired_by_block_saved_paths': paired_by_block_saved_paths,
    }